# Download and Inspect the Collection

The dataset was created from the Chronicling America collection — over 21 million digitized newspaper pages (1756–1963) curated by the Library of Congress and NEH. They used 39,330 pages (1800–1920), representing 53 US states, to ensure wide geographic and temporal coverage.

Source: https://dl.acm.org/doi/pdf/10.1145/3626772.3657891

GitHub: https://github.com/DataScienceUIBK/ChroniclingAmericaQA?tab=readme-ov-file

In [ ]:
!curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/test.json?download=true" -o test.json
!curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/train.json?download=true" -o train.json
!curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/dev.json?download=true" -o validation.json

import json

files = ["train.json", "validation.json", "test.json"]

for path in files:
    print(f"\n===== {path} =====")
    try:
        with open(path, "r", encoding="utf-8") as f:
            # Read a few hundred characters to see what kind of JSON it is
            head = f.read(500)
            print("Preview of first 500 characters:\n")
            print(head[:500])
        # Try to load only part of the file
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f"\nLoaded {len(data)} items (list).")
            print("Dictionary keys:", list(data[0].keys()))
            print(json.dumps(data[0], indent=2)[:600])
        elif isinstance(data, dict):
            print("\nTop-level is a dictionary. Keys:", list(data.keys()))
            for k, v in data.items():
                if isinstance(v, list):
                    print(f"Key '{k}' contains a list of {len(v)} items.")
                    if v:
                        print("First item keys:", list(v[0].keys()))
                        print(json.dumps(v[0], indent=2)[:600])
                        break
        else:
            print(f"Unexpected top-level type: {type(data)}")
    except Exception as e:
        print(f"Could not parse {path} as JSON: {e}")

# Create the Document Collection

To do that, we create a new json file that contains the 'para_id', 'context', 'raw_ocr', 'publication_date' keys, for all para_id in the collection.

para_id: is the id of a paragraph of a news paper page.

In [ ]:
import json
import os

inputs = ["train.json", "validation.json", "test.json"]
output = "document_collection.json"

def load_list_or_empty(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print(f"Skipping {path} because it is missing or empty")
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        print(f"Skipping {path} because it is not a list at the top level")
        return []
    except json.JSONDecodeError:
        print(f"Skipping {path} because it is not valid JSON")
        return []

def project(recs):
    out = []
    for r in recs:
        out.append({
            "para_id": r.get("para_id", ""),
            "context": r.get("context", ""),
            "raw_ocr": r.get("raw_ocr", ""),
            "publication_date": r.get("publication_date", "")
        })
    return out

all_recs = []
for p in inputs:
    recs = load_list_or_empty(p)
    print(f"Loaded {len(recs)} records from {p}")
    all_recs.extend(project(recs))

# deduplicate by para_id keeping the first one seen
uniq = {}
for rec in all_recs:
    pid = rec.get("para_id", "")
    if pid and pid not in uniq:
        uniq[pid] = rec

result = list(uniq.values())

with open(output, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(result)} records to {output}")
print(json.dumps(result[:3], indent=2))

# Create the Test Queries Data Structure

We keep the first 10.000 queries due to memory errors in the free colab version.

To be comparable, please keep the top 10.000 queries for evaluation.

In [ ]:
import json
import re
import unicodedata
import string

input_file = "test.json"
output_file = "test_queries.json"

# Load the data
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

def clean_question(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text)  # collapse multiple spaces
    return text.strip()

# Extract and clean
queries = [
    {
        "query_id": item.get("query_id", ""),
        "question": clean_question(item.get("question", "")),
    }
    for item in data
]

# Sort by query_id (assuming numeric)
queries = sorted(queries, key=lambda x: int(x["query_id"]) if str(x["query_id"]).isdigit() else x["query_id"])

# Keep only the first 10,000
queries = queries[:10000]

# Save new JSON
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)

print(f"Saved {len(queries)} entries to {output_file}")
print(json.dumps(queries[:3], indent=2))

# Create the Qrels for the test set

In [ ]:
input_file = "test.json"
qrels_file = "test_qrels.json"
answers_file = "test_query_answers.json"

# Load the data
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Build the qrels file: query_id, iteration=0, para_id, relevance=1
qrels = [
    {
        "query_id": item.get("query_id", ""),
        "iteration": 0,
        "para_id": item.get("para_id", ""),
        "relevance": 1
    }
    for item in data
]

# Build the query_answers file: same plus answer and org_answer
query_answers = [
    {
        "query_id": item.get("query_id", ""),
        "iteration": 0,
        "para_id": item.get("para_id", ""),
        "relevance": 1,
        "answer": item.get("answer", ""),
        "org_answer": item.get("org_answer", "")
    }
    for item in data
]

# Save both files
with open(qrels_file, "w", encoding="utf-8") as f:
    json.dump(qrels, f, ensure_ascii=False, indent=2)

with open(answers_file, "w", encoding="utf-8") as f:
    json.dump(query_answers, f, ensure_ascii=False, indent=2)

print(f"Saved {len(qrels)} entries to {qrels_file}")
print(f"Saved {len(query_answers)} entries to {answers_file}")
print("Sample qrels entry:", qrels[0])
print("Sample query_answers entry:", query_answers[0])

# **PHASE I:**

During the first phase of the project we define some baselines in order to compare the impact of the more advanced final pipelines.

In particular we focused on different ways of **indexing the documents:

  *   **Cleaned Text**: the first and best way of indexing that we found is all about using as only attribute the cleaned context given by the original paper. In addition we decide to pass also some temporal metadata as the publication date that will be useful also later during the PHASE II
  *   **Raw OCR**: in our fisrt proposal we talked about the possibility to analyze how effective was the cleaning done in the paper, so we use as a baseline also an indexing based on the raw OCR. Also in this case we decide to pass the publication date as a metadata but to give some more context to the algorithms we add also the clean text as an important metadata

Due to the computational limitations inherent to the standard Google Colab environment, specifically RAM and runtime restrictions, our try to insert and see how the collection reacts to full scale query expansion failed. Consequently, it led us to limiting the whole process to 2000 topic.

Finally, to run and evaluate this first phase of the project we used BM25 and TF-IDF separately as our baseline algorithms. We compare them using this metrics: P@1, P@5, P@10, R@5, R@10, nDCG@5, nDCG@10 and MAP.

In [ ]:
# Install and import all the necessary library

!pip install python-terrier

import json
import os
import shutil
import pyterrier as pt

## *Data preparation*

In [ ]:
# Change of format for the document collection in order to favour the indexing

with open("document_collection.json", "r", encoding="utf-8") as f:
    document_collection_data = json.load(f)

# we include 'publication_date' for later use in Phase II
def prepare_docs_for_indexing(data_list):
    for doc in data_list:
        new_doc = {
            "docno": doc.get("para_id", ""),
            "text": doc.get("context", ""),
            "publication_date": doc.get("publication_date", ""),
            "raw_ocr": doc.get("raw_ocr", "")
        }
        yield new_doc

## *Indexing strategies*

In [ ]:
# First indexing using the cleaned text

index_path_text = './text-index'

if os.path.exists(index_path_text):
    shutil.rmtree(index_path_text)

indexref_text = pt.IterDictIndexer(
    index_path_text,
    meta={'publication_date': 20, 'text': 4096, 'docno': 40},
    text_attrs=['text']
    ).index(prepare_docs_for_indexing(document_collection_data))

index_text = pt.IndexFactory.of(indexref_text)

stats_text = index_text.getCollectionStatistics()
print("Index folder:", index_path_text)
print("Number of documents:", stats_text.getNumberOfDocuments())
print("Number of postings:", stats_text.getNumberOfPostings())
print("Number of tokens:", stats_text.getNumberOfTokens())
print("Number of unique terms:", stats_text.getNumberOfUniqueTerms())
print("Average document length:", stats_text.getAverageDocumentLength())

In [ ]:
# Second indexing using the raw OCR extracted text

index_path_raw_ocr = './raw_ocr-index'

if os.path.exists(index_path_raw_ocr):
    shutil.rmtree(index_path_raw_ocr)

indexref_raw_ocr = pt.IterDictIndexer(
    index_path_raw_ocr,
    meta={'publication_date': 20, 'text': 4096, 'docno': 40},
    text_attrs=['raw_ocr']
    ).index(prepare_docs_for_indexing(document_collection_data))

index_raw_ocr = pt.IndexFactory.of(indexref_raw_ocr)

stats_raw = index_raw_ocr.getCollectionStatistics()
print("Index folder:", index_path_raw_ocr)
print("Number of documents:", stats_raw.getNumberOfDocuments())
print("Number of postings:", stats_raw.getNumberOfPostings())
print("Number of tokens:", stats_raw.getNumberOfTokens())
print("Number of unique terms:", stats_raw.getNumberOfUniqueTerms())
print("Average document length:", stats_raw.getAverageDocumentLength())

## *Exploratory Data Analysis*
The statistical analysis of the qrels reveals a strictly binary and sparse relevance distribution. Specifically, every query in the test set has exactly one relevant document: mean = 1.0, max = 1.0.
This characteristic, suggests that metric like P@1 will be a particularly significant indicator of performance, as finding that single relevant document is the primary goal.

In [ ]:
# Some important statistics of the queries

import pandas as pd

with open("test_qrels.json") as f:
    qrels = pd.DataFrame(json.load(f))

counts = qrels.groupby("query_id")["para_id"].count()
print("Overall Statistics")
print(counts.describe())

import matplotlib.pyplot as plt

plt.figure()
counts.plot(kind="hist")
plt.xlabel("Number of relevance assessments per query")
plt.ylabel("Number of queries")
plt.title("Relevance assessment distribution")
plt.show()

counts.sort_values(ascending=False).head()

# Count how many times each relevance label occurs overall
qrels["relevance"].value_counts()

plt.figure()
qrels["relevance"].plot(kind="hist")
plt.xlabel("Relevance score")
plt.ylabel("Frequency")
plt.title("Relevance score distribution")
plt.show()

## *Execution of baselines*


In [ ]:
# Change of format for the queries and the qrels in order to succsesfully run the pt.experiment

with open("test_queries.json", "r", encoding="utf-8") as f:
    topics = pd.DataFrame(json.load(f))

topics = topics.rename(columns={'query_id': 'qid', 'question': 'query'})
topics['qid'] = topics['qid'].astype(str)

with open("test_qrels.json", "r", encoding="utf-8") as f:
    qrels = pd.DataFrame(json.load(f))

qrels = qrels.rename(columns={'query_id': 'qid', 'para_id': 'docno', 'relevance': 'label'})
qrels['qid'] = qrels['qid'].astype(str)
qrels['docno'] = qrels['docno'].astype(str)
qrels['label'] = qrels['label'].astype(int)


In [ ]:
# Definition of the models used in the experiments

bm25_100_text = pt.terrier.Retriever(indexref_text, wmodel="BM25") %100
tfidf_text = pt.terrier.Retriever(indexref_text, wmodel="TF_IDF")
bm25_100_raw_ocr = pt.terrier.Retriever(indexref_raw_ocr, wmodel="BM25") %100
tfidf_raw_ocr = pt.terrier.Retriever(indexref_raw_ocr, wmodel="TF_IDF")

In [ ]:
# Run of the first experiment

print("Baseline experiment is running... (loading)")

experiment = pt.Experiment(
    [bm25_100_text, tfidf_text, bm25_100_raw_ocr, tfidf_raw_ocr],
    topics.head(2000),
    qrels,
    names=['BM25_text', 'TF-IDF_text', 'BM25_raw_ocr', 'TF-IDF_raw_ocr'],
    eval_metrics=['P.1', 'P.5', 'P.10', 'recall.5', 'recall.10', 'ndcg_cut.5', 'ndcg_cut.10', 'map'],
    batch_size = 25,
    verbose = True
)

print("\n=== RESULTS PHASE I ===")
display(experiment)

## *Results:*

The experimental results demonstrate a clear hierarchy in performance. BM25 on the cleaned text proves to be the most robust configuration, achieving a MAP of 0.69.
The comparison between clean text and raw OCR, highlights a significant gap, with performance dropping when using noisy data.
Given this findings, our team selected BM25 with clean text as the baseline retriever for the advanced pipelines in Phase II.

# **PHASE II: advanced retrieval archictectures**

In this second phase we focused on some more complex pipeline trying to improve the already good results of PHASE I. This involved two different types of experiments:

  *   **E1**: Semantic search (neural re-ranking)

  The first advanced experiment implements a neural re-ranking pipelline. After a baseline BM25 algorithm, we utilize a cross-encoder model (ms-marco-MiniLM-L-6-v2) to re-rank the top 10 documents retrieved, ensuring a more broad comprehension of the context. Unlike bi-encoders, cross-encoders process the query and docuemnt simultaneously, allowing to capture fine semantic relevance, beyond exact keyword matching.
  Due to the high computational cost of the cross-encoder, we limit a re-ranking to the top 10 documents of the query

  *  **E2**: Domain specific (temporal and entity scoring)
  
  In this second experiment we develop a custom re-ranker designed to address domain specific challenges. The scoring function integrates temporal relevance, by boosting documents published near the query's target date, and Named Entity Recognition, by penalizing documents that lack key entities mentioned in the query.

## **E1: Neural experiment**

In [ ]:
# neural re-ranking setup
!pip install -U sentence-transformers

from sentence_transformers import CrossEncoder

In [ ]:
# load pre-trained cross-encoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# scoring function applied per query
def cross_encoder_score_single_query(df):
    if df.empty:
        return df

    # create pairs
    pairs = [[row['query'], row['text']] for _, row in df.iterrows()]

    # predict score
    scores = model.predict(pairs, batch_size=len(df), show_progress_bar=False)

    df = df.copy()
    df['score'] = scores

    return df



In [ ]:
# creation neural pipeline

neural_model = pt.apply.by_query(cross_encoder_score_single_query)

get_text = pt.text.get_text(index_text, "text")

neural_pipe = bm25_100_text % 10 >> get_text >> neural_model

In [ ]:
# run part 1 on the second experiment

print("E1 is running... (loading)")

experiment = pt.Experiment(
    [bm25_100_text, neural_pipe],
    topics.head(2000),
    qrels,
    names=['BM25', 'E1: Neural'],
    eval_metrics=['P.1', 'P.5', 'P.10', 'recall.5', 'recall.10', 'ndcg_cut.5', 'ndcg_cut.10', 'map'],
    batch_size = 25,
    precompute_prefix = True,
    verbose = True
)

print("\n=== RESULTS PHASE 2: E1 ===")
display(experiment)

### *Results of E1:*
The neural re-ranker significantly improves P@1 and MAP compared to the BM25 pipeline. This confirms that semantic understanding helps bridge the vocabulary gap in historical texts.

## **E2: Custom experiment**
Here, we implement the custom re-ranker. This approach requires extracting years and entities from the text. We use spaCy for NER and regular expressions for temporal extractions.

In [ ]:
# E2 setup

!pip install -U spacy
!python -m spacy download it_core_news_sm

import spacy
import re

nlp = spacy.load('en_core_web_sm')

In [ ]:
# helper functions

# extraction of years 1800-1920 from text using regex
def extract_year(text):

  if not isinstance(text, str):
    return None
  match = re.search(r'\b(18\d{2}|19[0-2]\d)\b', text)
  if match is not None:
    return int(match.group(1))
  else:
    return None

# extracts the four digit year component from the document's publication date metadata
def doc_year(publ_date_str):
  if publ_date_str is None:
    return None

  s = str(publ_date_str).strip()

  if len(s) < 4:
    return None

  try:
    return int(s[:4])
  except ValueError:
    return None

In [ ]:
# scoring logic

# temporal
def temporal_score(row):

  q_year = extract_year(row['query'])
  d_year = doc_year(row['publication_date'])

  score = row['score']

  if q_year and d_year:
    diff = abs(q_year - d_year)
    if diff == 0:
      score *= 1.5
    elif diff <= 2:
      score *= 1.2
    elif diff <= 5:
      score *= 1.1

  return score

# NER
# checks if any entity in the query exists in the doc
def entities_match(query_entities, doc_entities):
  for e in query_entities:
    if e in doc_entities:
      return True
  return False

# penalizes score if named entities in query are missing in doc
def ner_score(row):
  query_nlp = nlp(row['query'])
  query_entities = [e.text for e in query_nlp.ents]
  doc_nlp = nlp(row['text'])
  doc_entities = [e.text for e in doc_nlp.ents]

  score = row['score']

  if not entities_match(query_entities, doc_entities):
    score *= 0.5

  return score

# final
# since both functions modify the base BM25 score, we just sum them up
def final_score(row):
  return temporal_score(row) + ner_score(row)

In [ ]:
# E2 pipeline construction
get_doc_properties = pt.text.get_text(index_text, ["text", "publication_date"])
final_pipe = bm25_100_text % 10 >> get_doc_properties >> pt.apply.doc_score(final_score)

# run part 2 of the second experiment
print("E2 is running... (loading)")

experiment_phase2 = pt.Experiment(
    [bm25_100_text, final_pipe],
    topics.head(2000),
    qrels,
    names=['BM25', 'Final Score'],
    eval_metrics=['P.1', 'P.5', 'P.10', 'recall.5', 'recall.10', 'ndcg_cut.5', 'ndcg_cut.10', 'map'],
    batch_size=25,
    precompute_prefix = True,
    verbose = True
)

print("\n=== RESULTS E2 ===")
display(experiment_phase2)

## **Conclusion on advanced experiments**

Both advanced strategies outperformed the BM25 baseline:

  *   **E1 (Neural)** achieved the highest performance, demonstrating the power of semantic matching, with MAP = 0.728 and P@1 = 0.678
  *   **E2 (Custom)** also showed improvement confirming that temporal metadata is a valuable signal for disambiguation, though less universally effective than the neural approach.
